In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input, RandomFlip, RandomRotation, RandomZoom
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import EfficientNetB0


class BatikClassifier:
    def __init__(self, img_size=(224, 224), batch_size=32, random_seed=42):
        self.img_size = img_size
        self.batch_size = batch_size
        self.random_seed = random_seed
        self.model = None
        self.base_model = None
        self.label_encoder = None
        self.class_names = None
        
    def load_data(self, data_dir, target_folder=None):
        """Load images recursively from subdirectories organized by class."""
        data_map = {}
        search_dir = os.path.join(data_dir, target_folder) if target_folder else data_dir
        
        if not os.path.exists(search_dir):
            raise ValueError(f"Directory not found: {search_dir}")
        
        for root, dirs, files in os.walk(search_dir):
            folder_name = os.path.basename(root)
            if folder_name.lower() in ['train', 'test', 'validation', 'val', target_folder or '']:
                continue
            
            image_files = [f for f in files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            if image_files:
                label = folder_name
                if label not in data_map:
                    data_map[label] = []
                for file in image_files:
                    file_path = os.path.join(root, file)
                    try:
                        img = cv2.imread(file_path)
                        if img is None:
                            continue
                        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        img = cv2.resize(img, self.img_size)
                        data_map[label].append(img)
                    except Exception:
                        continue
        
        if not data_map:
            raise ValueError("No valid images found in dataset")
        
        return data_map
    
    def prepare_data(self, data_map, test_size=0.2):
        """Prepare training and testing datasets."""
        X, y = [], []
        for label, images in data_map.items():
            X.extend(images)
            y.extend([label] * len(images))
        
        X = np.array(X, dtype=np.float32) / 255.0
        y = np.array(y)
        
        self.label_encoder = LabelEncoder()
        y_encoded = self.label_encoder.fit_transform(y)
        self.class_names = self.label_encoder.classes_
        y_categorical = to_categorical(y_encoded)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_categorical, test_size=test_size, 
            random_state=self.random_seed, stratify=y_encoded
        )
        
        return X_train, X_test, y_train, y_test
    
    def build_model(self, num_classes):
        """Build EfficientNetB0 based classification model with augmentation."""
        inputs = Input(shape=(*self.img_size, 3))
        
        x = RandomFlip("horizontal_and_vertical")(inputs)
        x = RandomRotation(0.2)(x)
        x = RandomZoom(0.1)(x)
        
        self.base_model = EfficientNetB0(
            weights='imagenet', include_top=False, input_tensor=x
        )
        self.base_model.trainable = False
        
        x = self.base_model.output
        x = GlobalAveragePooling2D()(x)
        x = Dropout(0.4)(x)
        x = Dense(256, activation='relu')(x)
        outputs = Dense(num_classes, activation='softmax')(x)
        
        self.model = Model(inputs=inputs, outputs=outputs)
        return self.model
    
    def train_phase1(self, X_train, y_train, X_test, y_test, epochs=15):
        """Train classifier head with frozen base model."""
        self.model.compile(
            optimizer=Adam(learning_rate=1e-3),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        return self.model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=epochs,
            batch_size=self.batch_size,
            callbacks=[EarlyStopping(patience=3, restore_best_weights=True, verbose=1)],
            verbose=1
        )
    
    def train_phase2(self, X_train, y_train, X_test, y_test, epochs=25, output_path='best_batik_model.keras'):
        """Fine-tune model by unfreezing top layers of base model."""
        self.base_model.trainable = True
        for layer in self.base_model.layers[:-20]:
            layer.trainable = False
        
        self.model.compile(
            optimizer=Adam(learning_rate=1e-5),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        
        callbacks = [
            ModelCheckpoint(output_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
            EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(factor=0.2, patience=3, min_lr=1e-7, verbose=1)
        ]
        
        return self.model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=epochs,
            batch_size=self.batch_size,
            callbacks=callbacks,
            verbose=1
        )
    
    def evaluate(self, X_test, y_test):
        """Evaluate model and return classification report."""
        try:
            self.model.load_weights("best_batik_model.keras")
        except Exception:
            pass
        
        y_pred_probs = self.model.predict(X_test, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.argmax(y_test, axis=1)
        
        report = classification_report(y_true, y_pred, target_names=self.class_names)
        cm = confusion_matrix(y_true, y_pred)
        
        return report, cm, y_true, y_pred
    
    def save_model(self, output_path='batik_model_deploy.h5'):
        """Save model for deployment without optimizer state."""
        self.model.save(output_path, include_optimizer=False)
        
        with open("labels.txt", "w") as f:
            f.write("\n".join(self.class_names))
        
        return output_path


def main():
    DATA_DIR = "path/to/your/dataset"
    TARGET_FOLDER = "raw_batik_v2.1"
    
    classifier = BatikClassifier()
    
    print("Loading data...")
    data_map = classifier.load_data(DATA_DIR, TARGET_FOLDER)
    
    print("Preparing datasets...")
    X_train, X_test, y_train, y_test = classifier.prepare_data(data_map)
    
    print("Building model...")
    classifier.build_model(num_classes=len(classifier.class_names))
    
    print("Phase 1: Training head...")
    classifier.train_phase1(X_train, y_train, X_test, y_test)
    
    print("Phase 2: Fine-tuning...")
    classifier.train_phase2(X_train, y_train, X_test, y_test)
    
    print("Evaluating model...")
    report, cm, y_true, y_pred = classifier.evaluate(X_test, y_test)
    print(report)
    
    print("Saving model...")
    classifier.save_model()
    
    print("Training complete.")


if __name__ == "__main__":
    main()